<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T00-L02-model-registry/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T00-L02-model-registry/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T00-L02-model-registry/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer. The cell below installs `pyyaml==6.0.3` and fetches
the files it needs beside it, and does nothing where they are already present. On Kaggle,
switch Internet on in the notebook's settings first; Kaggle allows that only for
phone-verified accounts.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = [("yaml", "pyyaml==6.0.3")]            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = ["assets/SOURCE.md", "assets/models.yaml", "assets/provider_snapshot.json"]    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T00-L02-model-registry/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# Build a role-based model registry, a retirement detector and a golden test that survives a model swap

**You will build:** a `models.yaml` resolver, a checker that diffs the registry against a dated
provider snapshot, and a regression test that still passes when the model behind a role changes.
**Time:** ~40 minutes · **Runs on:** a laptop CPU, 8 GiB RAM, offline · **Prerequisites:** T00-L01-the-8gb-track

By the end you will be able to:
1. Implement `resolve()` so a role such as `tier:api-cheap` follows `same_as` links to one pinned model id, and rejects a cycle.
2. Implement two checkers that flag a retired, missing, renamed or soon-to-retire pin, and a repriced or resized one.
3. Implement a golden-output regression test that compares a contract, not raw text, so swapping the model behind a role does not break it.
4. Measure, on a committed snapshot, how many pinned roles are already stale and how many golden cases survive a swap.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T00-L01`

**This lesson assumes you can already:**

- write Python functions that loop over dicts and lists of dicts, and return a NamedTuple
- read a nested YAML or JSON file and say which keys and values it holds
- parse an ISO date and add, subtract and compare days with the datetime module
- write a plain assert-based test for a function

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [The Python Tutorial, section 5: Data Structures](https://docs.python.org/3/tutorial/datastructures.html) · Python Software Foundation · *free* — dictionaries (5.5) and tuples (5.3) and how to loop over them, which is all that resolve() and the two checkers do
- [PyYAML documentation](https://pyyaml.org/wiki/PyYAMLDocumentation) · PyYAML project · *free* — the Tutorial section on loading a YAML document into Python dicts and lists, which is how models.yaml becomes the registry
- [datetime: Basic date and time types](https://docs.python.org/3/library/datetime.html) · Python Software Foundation · *free* — date, timedelta and date.fromisoformat, the pieces behind the notice-window arithmetic in the retirement detector
- [Software Engineering at Google, chapter 12: Unit Testing](https://abseil.io/resources/swe-book/html/ch12.html) · Winters, Manshreck and Wright (O'Reilly; free to read at abseil.io) · *free* — the sections on preventing brittle tests, testing via public APIs, and testing behaviours rather than methods: the reason the golden test compares a contract
- [Model deprecations (Claude Platform docs)](https://platform.claude.com/docs/en/about-claude/model-deprecations) · Anthropic · *free* — a real retirement table, with states, tentative retirement dates and replacements, which is what the provider snapshot in this lesson mirrors

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import copy
import io
import json
import sys
import traceback
from datetime import date, timedelta
from pathlib import Path
from typing import Callable, NamedTuple

import yaml

print("python", sys.version.split()[0], "· pyyaml", yaml.__version__)


def asset_path(name: str) -> Path:
    """Locate a file that ships in this lesson's assets/ folder. There is no network branch."""
    try:
        here = Path(__file__).resolve().parent
    except NameError:  # a notebook has no __file__
        here = Path.cwd()
    for candidate in (here / "assets" / name, here.parent / "assets" / name,
                      Path.cwd() / "assets" / name):
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"{name} not found in assets/ beside this lesson; it is never downloaded "
                            "by the lesson code (the launcher cell fetches it when it is missing).")


REGISTRY: dict = yaml.safe_load(asset_path("models.yaml").read_text())
SNAPSHOT: dict = json.loads(asset_path("provider_snapshot.json").read_text())
AS_OF = date.fromisoformat(SNAPSHOT["snapshot_date"])   # "today" for every check: never the wall clock
NOTICE_DAYS = 60   # the provider's stated minimum notice before retiring a public model (claims.yaml)

print(f"registry: {len(REGISTRY['roles'])} roles in models.yaml")
print(f"snapshot: {len(SNAPSHOT['models'])} models from provider '{SNAPSHOT['provider']}', taken {AS_OF}")
print(f"every check below treats {AS_OF} as today, so the output is the same on any night")


class Finding(NamedTuple):
    """One thing the nightly job wants a human to look at."""
    role: str      # the role that owns the pin
    kind: str      # renamed | retired | deprecated | missing | retiring_soon | repriced | context_changed | cutoff_changed
    model: str     # the provider's canonical id the finding is about
    detail: str


class Pin(NamedTuple):
    """What a role resolves to."""
    role: str                  # the role that was asked for
    chain: tuple               # roles followed, starting at `role`, ending at the role that holds the pin
    model: str                 # the pinned model id
    expected: dict             # what the registry recorded about it: prices, context window, cut-off


_FAILED_CHECKS: list[str] = []

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("resolve",),
    "exercise 2": ("lifecycle_findings",),
    "exercise 3": ("drift_findings",),
    "exercise 4": ("normalise_answer",),
    "exercise 5": ("golden_regression",),
}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (drift_findings)"; several -> "exercises 3, 6 and 8"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message, which names the likely mistake, and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped: needs {_named(waiting)} to pass first.")
        return
    try:
        check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped: needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet: fill in {stub}() above, then re-run this cell.")
        else:
            print(f"{label}: skipped: {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED: {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. Why a tutorial should name a role, not a model

A model id is the fastest-rotting string in a lesson. Providers retire models on a schedule,
reprice them, and ship a successor with a different context window. Every notebook
that hard-codes an id breaks the same week, and nobody knows which ones.

The fix is one file. Notebooks ask for a **role** (`tier:api-cheap`); `models.yaml` says which
model id sits behind it today, plus what we *believe* about that model: its price, context
window and cut-off. A scheduled job then compares those beliefs with a fresh copy of the
provider's list and opens an issue when they no longer match.

Here the fresh copy is a **committed snapshot**, so this notebook runs offline and gives the
same answer on any day. A live job would replace one function, the one that reads the file.
The cell below shows the registry as written: a fixture that was last checked long ago.

In [ ]:
print("role".ljust(22), "pins / follows".ljust(28), "recorded USD per Mtok in/out")
for _role, _entry in REGISTRY["roles"].items():
    _target = _entry.get("pin") or ("-> " + _entry["same_as"])
    _cost = f"{_entry.get('input_usd_per_mtok', '?')}/{_entry.get('output_usd_per_mtok', '?')}"
    print(_role.ljust(22), _target.ljust(28), _cost)

_by_status: dict[str, int] = {}
for _m in SNAPSHOT["models"]:
    _by_status[_m["status"]] = _by_status.get(_m["status"], 0) + 1
print("\nsnapshot models by lifecycle status:", dict(sorted(_by_status.items())))

## 2. Exercise 1: resolve a role to a pin

A role either holds a `pin` (a model id plus what we recorded about it) or says
`same_as: <another role>`. Follow the links until you reach a pin. A role that holds a `pin`
ends the walk, even if it also says `same_as`.

Two things go wrong in real registries: a role name that is not there (a typo in a
notebook), and roles that point at each other after an edit. A checker that loops forever
or dies with a stack overflow at 02:00 is worse than no checker, so both must raise a clear
error naming the role or the loop.

<details><summary>💡 Hint 1: what to think about</summary>

How do you know you have arrived? What would tell you that you are going round in a circle?
Keep a record of every role you have already visited.
</details>

<details><summary>💡 Hint 2: the approach, in words</summary>

Walk from the requested role. At each step: the role must exist; it must not be one you have
already seen; if it has a pin you are done, otherwise follow its `same_as`. The
`expected` dictionary is everything the pinning role records except the pin itself.
</details>

In [ ]:
def resolve(registry: dict, role: str) -> Pin:
    """Follow `same_as` links from `role` to the role that holds a `pin`.

    Example:
        >>> reg = {"roles": {"a": {"pin": "m-1", "input_usd_per_mtok": 2}, "b": {"same_as": "a"}}}
        >>> resolve(reg, "b")
        Pin(role='b', chain=('b', 'a'), model='m-1', expected={'input_usd_per_mtok': 2})

    Returns a Pin. Raises KeyError for an unknown role, and ValueError for a cycle or for a
    role that has neither `pin` nor `same_as`. Never modifies `registry`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


class _Watched(dict):
    """A roles table that counts lookups. A walk that goes round in a circle then fails the
    check with a message, instead of hanging the notebook with a list that never stops growing."""
    LIMIT = 1000

    def _tick(self) -> None:
        self.lookups = getattr(self, "lookups", 0) + 1
        if self.lookups > self.LIMIT:
            raise AssertionError(f"resolve() made {self.LIMIT} lookups in a registry of {len(self)} roles: it is "
                                 "going round in a circle. Remember every role you have visited and stop when "
                                 "you meet one again, wherever the circle starts")

    def __getitem__(self, key):
        self._tick()
        return super().__getitem__(key)

    def __contains__(self, key):
        self._tick()
        return super().__contains__(key)

    def get(self, key, default=None):
        self._tick()
        return super().get(key, default)


# Public checks: run them as often as you like.
def _check_resolve() -> None:
    reg = {"roles": {"a": {"pin": "m-1", "input_usd_per_mtok": 2},
                     "b": {"same_as": "a"}, "c": {"same_as": "b"}}}
    got = resolve(reg, "c")
    assert got.model == "m-1", "resolve(reg, 'c') should follow c -> b -> a and return a's pin"
    assert got.chain == ("c", "b", "a"), "chain should list every role visited, starting with the one asked for"
    assert got.expected == {"input_usd_per_mtok": 2}, "expected is the pinning role's record without the 'pin' key"
    got.expected["input_usd_per_mtok"] = 99
    assert reg["roles"]["a"]["input_usd_per_mtok"] == 2, "do not hand back the registry's own dict; the caller may edit it"
    try:
        resolve(reg, "nope")
    except KeyError:
        pass
    else:
        raise AssertionError("an unknown role should raise KeyError")
    loop = {"roles": _Watched({"x": {"same_as": "y"}, "y": {"same_as": "x"}})}
    try:
        resolve(loop, "x")
    except ValueError:
        pass
    except RecursionError:
        raise AssertionError("x and y point at each other: detect the cycle and raise ValueError, do not recurse forever") from None
    else:
        raise AssertionError("a cycle of same_as links should raise ValueError")
    for name in REGISTRY["roles"]:
        assert resolve(REGISTRY, name).model, f"role {name} should resolve to a model id"
    print("exercise 1 looks right")


_try("exercise 1", _check_resolve)

## 3. The snapshot, and what "retired" means

`provider_snapshot.json` holds one record per model id: its lifecycle `status`, an optional
`retirement` date, its recorded prices, context window and cut-offs. **A `null` means the
snapshot does not know, and unknown is not the same as changed.**

Two details of the provider's own scheme matter, and both come from its published pages
(see `claims.yaml`). Some ids are convenience **aliases** that point at the newest dated
snapshot; the snapshot's `aliases` table lists the ones the provider's pages name, so a pin on an
alias the table does not list is reported `missing` until someone adds it. And a retirement date on
an active model is often a floor: `retirement_is_earliest` says the model will not go before it, not
that it goes on it. The provider also promises a minimum notice before any retirement (`NOTICE_DAYS`
in the setup cell), so an active model whose floor has passed with no deprecation announced still has
at least that long: a warning about it is an early reminder, not an emergency.

The snapshot is dated, and the live table has moved on since (as of 14 October 2026, see `claims.yaml`):
the provider has deprecated the model behind `tier:api-previous`, whose floor had just passed, with a
retirement date no sooner than that notice allows, and has added a new Haiku model. This notebook keeps
the dated copy so that its answers never change under you; a live nightly job would have reported the
deprecation on its next run.

In [ ]:
print("aliases in the snapshot:", SNAPSHOT["aliases"])
_soon = sorted((m["retirement"], m["id"]) for m in SNAPSHOT["models"]
               if m["status"] == "active" and m["retirement"] and m["retirement_is_earliest"])[:3]
print("the three earliest floors on active models:")
for _when, _mid in _soon:
    print(f"  {_mid:<30} {_when}  ({(date.fromisoformat(_when) - AS_OF).days} days after the snapshot date)")

## 4. Exercise 2: lifecycle findings for one pin

Given one pinned id, say what the snapshot thinks of it. Look the id up in the model table first;
only an id that is not there is looked up in the alias table. Kinds, in the order you report them:

- `missing`: the id is not in the model table and is not a listed alias, or it is an alias whose
  target is not in the model table. Return just this finding.
- `renamed`: the id is an alias. Report it first and keep it, whatever the findings that follow, then
  carry on with the model it resolves to.
- `retired`: status is retired. The detail names the provider's replacement when there is
  one. A retired model gets no further lifecycle findings.
- `deprecated`: status is deprecated.
- `retiring_soon`: a retirement date is at most `warn_days` days after `as_of`. The detail
  contains the number of days. A date already in the past counts as soon, however long ago it was.

The `model` field of every finding except `missing` is the provider's canonical id.

<details><summary>💡 Hint 1: what to think about</summary>

What should happen when an alias points at an id the model table does not have? When an alias
points at a retired model, what does the list hold at the end? And is "exactly `warn_days` days
away" soon or not yet? What about a date that passed long ago?
</details>

<details><summary>💡 Hint 2: the approach, in words</summary>

Build a lookup by id. If the id is absent, try the alias table; give up with `missing`
otherwise. Collect findings in a list in the order above. Days remaining is the retirement
date minus `as_of`, using `date.fromisoformat`.
</details>

In [ ]:
def lifecycle_findings(role: str, model: str, snapshot: dict, as_of: date,
                       warn_days: int = NOTICE_DAYS) -> list:
    """Lifecycle findings for `model`, pinned by `role`, judged against `snapshot` on `as_of`.

    Example (a model that retires 10 days after as_of, with a 30-day window):
        >>> lifecycle_findings("r", "m", snap, date(2030, 1, 1), warn_days=30)
        [Finding(role='r', kind='retiring_soon', model='m', detail='10 days to the retirement date 2030-01-11')]

    Returns a list of Finding, empty when the model is healthy. Never raises for an unknown id.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _toy_snapshot() -> dict:
    keys = ("deprecated", "replacement", "input_usd_per_mtok", "output_usd_per_mtok",
            "context_tokens", "reliable_cutoff", "training_cutoff")
    def rec(mid, status, retirement=None, earliest=False, **kw):
        base = {"id": mid, "status": status, "retirement": retirement,
                "retirement_is_earliest": earliest, **{k: None for k in keys}}
        return {**base, **kw}
    return {"aliases": {"m-alias": "m-new"},
            "models": [rec("m-new", "active", "2030-01-11", True), rec("m-old", "retired", "2029-05-01",
                       replacement="m-new"), rec("m-far", "active", "2035-01-01", True)]}


def _check_lifecycle() -> None:
    snap, today = _toy_snapshot(), date(2030, 1, 1)
    kinds = lambda mid, days=30: [f.kind for f in lifecycle_findings("r", mid, snap, today, days)]
    assert kinds("m-far") == [], "a healthy model far from retirement has no findings"
    assert kinds("m-ghost") == ["missing"], "an id that is neither a model nor an alias is 'missing'"
    old = lifecycle_findings("r", "m-old", snap, today)
    assert [f.kind for f in old] == ["retired"], "a retired model is 'retired' only: no retiring_soon on top"
    assert "m-new" in old[0].detail, "the retired finding's detail should name the recommended replacement"
    assert kinds("m-new") == ["retiring_soon"], "ten days away inside a thirty-day window should be retiring_soon"
    assert kinds("m-alias") == ["renamed", "retiring_soon"], "an alias is 'renamed', then judged as the model it points at"
    assert lifecycle_findings("r", "m-alias", snap, today)[0].model == "m-new", "report the canonical id, not the alias"
    print("exercise 2 looks right")


_try("exercise 2", _check_lifecycle)

## 5. Exercise 3: drift findings for one pin

A model can be perfectly healthy and still not be the model you wrote down. Compare what the
registry recorded (`expected`) with what the snapshot says:

- `repriced`: input or output price differs. ONE finding per model, its detail naming what changed.
- `context_changed`: `context_tokens` differs.
- `cutoff_changed`: `reliable_cutoff` differs.

Any difference counts, however small. A field that is `None` on either side is *unknown*, so it never
produces a finding; a recorded `0` is a value, not an unknown. An id that is
an alias is compared against the model it resolves to; an id the snapshot does not know
produces nothing here (exercise 2 already reports it). Report in the order above.

<details><summary>💡 Hint 1: what to think about</summary>

What does it mean for the snapshot to say `null`? Should a registry that never recorded a cut-off
be told that its cut-off is wrong? And if both prices changed, is that one problem or two?
</details>

<details><summary>💡 Hint 2: the approach, in words</summary>

Find the record (through the alias table if you must). Collect the price differences whose
two sides are both known into one list; emit one finding if the list is non-empty. Do the same
for each of the other two fields.
</details>

In [ ]:
def drift_findings(role: str, model: str, expected: dict, snapshot: dict) -> list:
    """Compare what the registry recorded for `model` with what `snapshot` says now.

    Example (registry recorded an input price of 3, the snapshot says 2):
        >>> drift_findings("r", "m", {"input_usd_per_mtok": 3}, snap)
        [Finding(role='r', kind='repriced', model='m', detail='USD per Mtok: input 3 -> 2')]

    Returns a list of Finding in the order repriced, context_changed, cutoff_changed.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_drift() -> None:
    snap = _toy_snapshot()
    snap["models"][0].update(input_usd_per_mtok=2, output_usd_per_mtok=10, context_tokens=1000, reliable_cutoff="2026-06")
    kinds = lambda exp, mid="m-new": [f.kind for f in drift_findings("r", mid, exp, snap)]
    assert kinds({"input_usd_per_mtok": 2, "output_usd_per_mtok": 10, "context_tokens": 1000}) == [], \
        "recorded values that match the snapshot are not drift"
    assert kinds({"input_usd_per_mtok": 3, "output_usd_per_mtok": 15}) == ["repriced"], \
        "a change to input AND output is one 'repriced' finding, not two"
    assert kinds({"output_usd_per_mtok": 15}) == ["repriced"], "a change to the output price alone still counts"
    assert kinds({"context_tokens": 200, "reliable_cutoff": "2025-02"}) == ["context_changed", "cutoff_changed"], \
        "report context_changed before cutoff_changed"
    assert kinds({"input_usd_per_mtok": 3}, "m-far") == [], \
        "the snapshot has no price for m-far: unknown is not 'changed'"
    assert kinds({}) == [], "a registry that recorded nothing cannot be wrong about it"
    assert kinds({"input_usd_per_mtok": 3}, "m-alias") == ["repriced"], "an alias is compared against the model it resolves to"
    assert kinds({"input_usd_per_mtok": 3}, "m-ghost") == [], "an unknown id is exercise 2's finding, not drift"
    print("exercise 3 looks right")


_try("exercise 3", _check_drift)

## 6. The nightly job, end to end

The two checkers are the core. Around them sit three small pieces that are **given to you**: `audit()`
runs both over every pinned role, `exit_code()` turns findings into a process exit status
(2 for a model that is gone, 1 for anything else worth a look, 0 for clean), and `new_findings()` keeps
the job from re-opening yesterday's issue every night. It matches findings by role, kind and model,
not by their detail, because a detail can carry a day count that changes every night.
A role that only says `same_as` owns no
pin, so it owns no findings: one retired model should open one issue, not three.

In [ ]:
FAIL_KINDS = {"retired", "missing"}


def audit(registry: dict, snapshot: dict, as_of: date, warn_days: int = NOTICE_DAYS) -> list:
    """Every finding for every role that holds a pin, in role-name order. Given to you."""
    found: list = []
    for role in sorted(registry["roles"]):
        pin = resolve(registry, role)
        if len(pin.chain) > 1:      # a follower: the role it follows owns the findings
            continue
        found += lifecycle_findings(role, pin.model, snapshot, as_of, warn_days)
        found += drift_findings(role, pin.model, pin.expected, snapshot)
    return found


def exit_code(findings: list) -> int:
    """2 if any pinned model is retired or missing, 1 for any other finding, else 0. Given to you."""
    if any(f.kind in FAIL_KINDS for f in findings):
        return 2
    return 1 if findings else 0


def new_findings(before: list, after: list) -> list:
    """Findings present now that were not there last night. Given to you.

    The same issue on two nights has the same role, kind and model; its detail may differ (a day
    count moves every night), so the detail is left out of the comparison.
    """
    seen = {(f.role, f.kind, f.model) for f in before}
    return [f for f in after if (f.role, f.kind, f.model) not in seen]


def _show_audit() -> None:
    findings = audit(REGISTRY, SNAPSHOT, AS_OF)
    pinned = [r for r in REGISTRY["roles"] if len(resolve(REGISTRY, r).chain) == 1]
    followers = [r for r in REGISTRY["roles"] if r not in pinned]
    print(f"audit of {len(pinned)} pinned roles ({len(followers)} more only follow another), as of {AS_OF}:")
    for f in findings:
        print(f"  {f.kind:<15} {f.role:<20} {f.model:<27} {f.detail}")
    dirty = sorted({f.role for f in findings})
    print(f"\n{len(dirty)} of {len(pinned)} pinned roles have at least one finding; exit code {exit_code(findings)}")
    print("healthy roles:", ", ".join(r for r in pinned if r not in dirty) or "none")


_try("nightly audit", _show_audit, needs=("exercise 1", "exercise 2", "exercise 3"))

### A second night, on purpose

Yesterday's report is not tomorrow's. The cell below moves "today" on by one day, first with the same
snapshot and then with a **synthetic** copy in which the provider deprecates the model behind
`tier:api-strong` and announces a retirement date inside the notice window. Nothing here is real: it is
a what-if, built in memory. Watch which findings come back as new, and which stay quiet although their
day counts moved.

In [ ]:
def _show_second_night() -> None:
    tonight = AS_OF + timedelta(days=1)
    tomorrow = copy.deepcopy(SNAPSHOT)
    strong = resolve(REGISTRY, "tier:api-strong").model
    for m in tomorrow["models"]:
        if m["id"] == strong:
            m.update(status="deprecated", deprecated=str(tonight),
                     retirement=str(tonight + timedelta(days=NOTICE_DAYS // 2)), retirement_is_earliest=False)
    before = audit(REGISTRY, SNAPSHOT, AS_OF)
    same = audit(REGISTRY, SNAPSHOT, tonight)
    moved = sum(f not in before for f in same)
    print(f"last night ({AS_OF}): {len(before)} findings")
    print(f"tonight ({tonight}), same snapshot: {len(same)} findings, {moved} with a changed detail, "
          f"{len(new_findings(before, same))} new")
    after = audit(REGISTRY, tomorrow, tonight)
    fresh = new_findings(before, after)
    print(f"tonight, synthetic snapshot: {len(after)} findings, {len(fresh)} new")
    for f in fresh:
        print(f"  NEW {f.kind:<14} {f.role}: {f.detail}")
    print(f"exit code over every finding: {exit_code(after)} · over the new ones alone: {exit_code(fresh)}")


_try("second night", _show_second_night, needs=("exercise 1", "exercise 2", "exercise 3"))

## 7. A golden test that survives a swap

Pinned ids are stable, but the pin behind a role will change: a retirement forces it. Then you need
to know whether your lesson still works. A **golden-output test** records the answers a human approved
and re-checks them on every run. Compare raw strings and it fails the moment a new model words the
same answer differently: formatting, key order, capitals, a code fence. Compare only the *contract*
(the fields your code reads, normalised) and it passes exactly when the behaviour you depend on survived.

There is no model in this lesson. **Simulated** backends, plain Python functions with made-up
ids, stand in for real models answering the same ticket-triage prompt. `sim-model-a` answers compact JSON, `sim-model-b`
answers in a different style, and `sim-model-c` has regressed on some tickets.

In [ ]:
def _truth(prompt: str) -> tuple:
    """The simulated 'ground truth' every backend starts from: (priority, team)."""
    p = prompt.lower()
    team = "billing" if ("charged" in p or "invoice" in p) else "account" if "password" in p else "engineering"
    if "not urgent" in p or "no rush" in p:
        prio = "low"
    elif "urgent" in p or " down" in p:
        prio = "high"
    else:
        prio = "medium" if ("crash" in p or "error" in p) else "low"
    return prio, team


def _sim_a(prompt: str) -> str:
    prio, team = _truth(prompt)
    return json.dumps({"priority": prio, "team": team})


def _sim_b(prompt: str) -> str:
    prio, team = _truth(prompt)
    return f'```json\n{{\n  "team": "{team.title()}",\n  "priority": " {prio.upper()} ",\n  "confidence": 0.9\n}}\n```'


def _sim_c(prompt: str) -> str:
    prio, team = _truth(prompt)
    if team == "account":
        team = "billing"                    # regression 1: password tickets go to the wrong team
    if " down" in prompt.lower():
        prio = "medium"                     # regression 2: an outage is under-prioritised
    if "not urgent" in prompt.lower():
        return f"The priority is {prio}."   # regression 3: prose instead of JSON
    return json.dumps({"priority": prio, "team": team})


SIM_BACKENDS = {"sim-model-a": _sim_a, "sim-model-b": _sim_b, "sim-model-c": _sim_c}
KEYS = ("priority", "team")
CASES = [
    {"id": "t1", "prompt": "I was charged twice for one order and it is urgent"},
    {"id": "t2", "prompt": "the app crashes when I open settings"},
    {"id": "t3", "prompt": "I forgot my password, how do I reset it?"},
    {"id": "t4", "prompt": "checkout is down for all our customers"},
    {"id": "t5", "prompt": "please update the invoice address, no rush"},
    {"id": "t6", "prompt": "error 500 when exporting, not urgent"},
]
# Recorded once from an approved run and reviewed by a human: the contract for each case.
GOLDEN = {
    "t1": {"priority": "high", "team": "billing"},
    "t2": {"priority": "medium", "team": "engineering"},
    "t3": {"priority": "low", "team": "account"},
    "t4": {"priority": "high", "team": "engineering"},
    "t5": {"priority": "low", "team": "billing"},
    "t6": {"priority": "low", "team": "engineering"},
}
DEMO_REGISTRY = {"version": 1, "roles": {"tier:demo-triage": {"pin": "sim-model-a"}}}


def swap_pin(registry: dict, role: str, new_model: str) -> dict:
    """A copy of `registry` with `role` re-pinned: the one-line edit a retirement forces. Given to you."""
    swapped = copy.deepcopy(registry)
    swapped["roles"][role]["pin"] = new_model
    return swapped


print("raw output of the same prompt from two backends:")
print("  a:", _sim_a(CASES[0]["prompt"]))
print("  b:", _sim_b(CASES[0]["prompt"]).replace("\n", "\\n"))

## 8. Exercise 4: normalise an answer to its contract

Turn one raw model reply into the small dictionary your code depends on:

1. Strip surrounding whitespace. If the reply is wrapped in a Markdown code fence, take what is inside it.
2. Parse the JSON. Anything that is not a JSON object raises `ValueError`.
3. Keep only the requested `keys`, in that order. A missing key raises `ValueError`.
4. Text values are lower-cased with runs of whitespace collapsed to single spaces. Other values are kept as they are.

<details><summary>💡 Hint 1: what to think about</summary>

What are all the ways two correct answers can differ without differing in meaning? Which of your
steps removes each? And what should a broken reply do: crash the whole test run, or count as a failure?
</details>

<details><summary>💡 Hint 2: the approach, in words</summary>

A fenced reply starts with a line of backticks (maybe with a language tag) and ends with another. Drop those
two lines. `json.JSONDecodeError` is a subclass of `ValueError`, but say what went wrong in your own message.
`str.split()` called with no argument splits on any run of whitespace, tabs and newlines included.
</details>

In [ ]:
def normalise_answer(text: str, keys: tuple) -> dict:
    """Reduce a raw reply to {key: normalised value} for the requested keys.

    Example:
        >>> normalise_answer('```json\\n{"team": " Billing ", "priority": "HIGH", "x": 1}\\n```', ("priority", "team"))
        {'priority': 'high', 'team': 'billing'}

    Returns a dict. Raises ValueError for text that is not a JSON object or lacks a requested key.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_normalise() -> None:
    fenced = '```json\n{"team": " Billing  Ops ", "priority": "HIGH", "extra": 1}\n```'
    assert normalise_answer(fenced, KEYS) == {"priority": "high", "team": "billing ops"}, \
        "fenced JSON should be unwrapped, then text lower-cased and whitespace collapsed, extra keys dropped"
    assert normalise_answer('{"priority": "low", "team": "x"}', KEYS) == {"priority": "low", "team": "x"}, \
        "plain JSON should work without a fence"
    assert normalise_answer('{"n": 3, "ok": true}', ("n", "ok")) == {"n": 3, "ok": True}, \
        "only text is normalised: numbers and booleans must come back unchanged"
    for bad in ("The priority is low.", '["low"]', '{"priority": "low"}'):
        try:
            normalise_answer(bad, KEYS)
        except ValueError:
            continue
        raise AssertionError(f"{bad!r} is not a usable answer: it should raise ValueError")
    print("exercise 4 looks right")


_try("exercise 4", _check_normalise)

## 9. Exercise 5: the golden regression, through a role

Run every case through the model **the role resolves to right now**, normalise each reply, and compare
with the golden contract. The test must never name a model id: the whole point is that the registry
edit is the only thing that changes when a model is swapped.

A reply that cannot be normalised is a failed case, not a crashed test run.

<details><summary>💡 Hint 1: what to think about</summary>

Which earlier exercise tells you which backend to call? What should the report say about *which* model
actually answered, so a failing run is diagnosable?
</details>

<details><summary>💡 Hint 2: the approach, in words</summary>

Resolve the role once, look the model up in `backends`, and loop over the cases in order. For each one call the
backend, normalise, compare with `golden[case id]`, and record the id in `passed` or `failed`.
</details>

In [ ]:
def golden_regression(registry: dict, role: str, backends: dict, cases: list,
                      golden: dict, keys: tuple) -> dict:
    """Run `cases` through whichever model `role` resolves to and compare with `golden`.

    Example (a two-case run where the model gets the second wrong):
        >>> golden_regression(reg, "tier:x", backends, cases, golden, ("k",))
        {'model': 'sim-model-a', 'passed': ['c1'], 'failed': ['c2']}

    Returns {"model": resolved id, "passed": [case ids], "failed": [case ids]}, both lists in case order.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_golden() -> None:
    rep = golden_regression(DEMO_REGISTRY, "tier:demo-triage", SIM_BACKENDS, CASES, GOLDEN, KEYS)
    assert rep["model"] == "sim-model-a", "report the id the role resolved to"
    assert rep["failed"] == [] and rep["passed"] == [c["id"] for c in CASES], \
        "the approved model must pass every case, listed in case order"
    swapped = swap_pin(DEMO_REGISTRY, "tier:demo-triage", "sim-model-b")
    rep = golden_regression(swapped, "tier:demo-triage", SIM_BACKENDS, CASES, GOLDEN, KEYS)
    assert rep["model"] == "sim-model-b" and rep["failed"] == [], \
        "after re-pinning the role the test must follow it, and a differently worded but equivalent model must pass"
    print("exercise 5 looks right")


_try("exercise 5", _check_golden)

### The swap, measured

Now re-pin the role three times and count. The **exact** column compares the raw text with the approved
model's raw text, which is what a naive golden file does. The **contract** column is your test.

In [ ]:
def _show_swap() -> None:
    approved = {c["id"]: SIM_BACKENDS["sim-model-a"](c["prompt"]) for c in CASES}
    print(f"{'pin behind the role':<20} {'exact-text match':<18} {'contract match':<16} failed cases")
    rescued, caught = [], []
    for pin in SIM_BACKENDS:
        reg = swap_pin(DEMO_REGISTRY, "tier:demo-triage", pin)
        rep = golden_regression(reg, "tier:demo-triage", SIM_BACKENDS, CASES, GOLDEN, KEYS)
        exact = sum(SIM_BACKENDS[pin](c["prompt"]) == approved[c["id"]] for c in CASES)
        print(f"{rep['model']:<20} {exact} of {len(CASES):<12} {len(rep['passed'])} of {len(CASES):<10} {rep['failed'] or 'none'}")
        if not rep["failed"] and exact < len(CASES):
            rescued.append(pin)
        if rep["failed"]:
            caught.append(pin)
    print("\nswaps the contract test passes and a raw-text test would have rejected:", ", ".join(rescued) or "none")
    print("swaps the contract test fails:", ", ".join(caught) or "none")


_try("the swap", _show_swap, needs=("exercise 1", "exercise 4", "exercise 5"))

## 10. Running it nightly, and what it cannot see

On GitHub the nightly job is a workflow with a `schedule:` trigger that runs the audit and opens an issue
when `exit_code` is not zero. Not run here, and its shape is only this:

```yaml
on:
  schedule:
    - cron: "17 3 * * *"     # off the hour: the top of every hour is when scheduled runs queue up
jobs:
  models:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v7       # the repository, so the job can read models.yaml
      - run: python check_models.py   # your audit, fed a fresh copy of the provider's list
```

Two limits, both from GitHub's documentation (`claims.yaml`): scheduled workflows run only on the default
branch, and in a public repository they are disabled after a period with no repository activity. A quiet
repo can silently lose its own alarm.

And what the diff cannot see: it compares ids, prices, limits and dates. The provider says the weights behind
an id do not change, but that the serving infrastructure can, with minor differences in behaviour. Only the
golden test notices that, which is why you need both.

In [ ]:
def _tonight() -> None:
    findings = audit(REGISTRY, SNAPSHOT, AS_OF)
    by_kind: dict[str, int] = {}
    for f in findings:
        by_kind[f.kind] = by_kind.get(f.kind, 0) + 1
    code = exit_code(findings)
    print("findings by kind:", dict(sorted(by_kind.items())))
    print(f"the job would exit {code}:", {0: "green", 1: "open an issue", 2: "open an issue and fail the run"}[code])


_try("tonight's job", _tonight, needs=("exercise 1", "exercise 2", "exercise 3"))

## 11. Common mistakes

- **Treating `null` as "changed".** A field the snapshot does not know is unknown. Flagging it buries the real findings.
- **Comparing raw text in a golden test.** It fails on a harmless rewording and teaches the team to ignore the alarm.
- **Letting a broken reply crash the run.** A regression is a failed case with a name, not a traceback.
- **Judging a retirement date without saying which kind it is.** A floor is not a schedule.
- **Reading the wall clock.** A check that depends on today cannot be re-run on last night's snapshot.

## 12. Self-check

1. A provider table gives an active model a tentative retirement date, "not sooner than" some day well beyond your warning window.
   What should the nightly job do today?
   - (a) Fail the build now, because retirement is coming. (b) Ignore the entry for ever, because "tentative" means it may never happen.
   - (c) Stay quiet for now and start warning once the date is inside the window. (d) Swap in the newest model immediately.
2. Your golden test compares raw strings, and it fails after a swap although every answer means the same. What keeps the test useful?
   - (a) Compare only the normalised fields your code reads. (b) Assert that the reply is not empty.
   - (c) Regenerate the golden file from the new model on every run. (d) Skip the test whenever the pin changes.
3. Which of these can a diff of ids, prices, limits and dates NOT detect?
   - (a) A model has been retired. (b) A model has been repriced. (c) A pin has a typo.
   - (d) The provider changed how a model behaves behind an unchanged id.
4. Why do lessons name roles such as `tier:api-cheap` instead of model ids?
   - (a) Model ids are secret. (b) A retirement is fixed in one file, and the golden test shows whether the swap kept the behaviour.
   - (c) Roles make the API cheaper. (d) Providers require it.

Answers come with this lesson's worked solution when you enrol on Synapsa.

## What you built, and where it goes next

`models.yaml` plus the checker is the freshness policy for the rest of this course: any lesson that
needs a model asks for a role and the registry says which id it is today. When the nightly job goes
red, one edit to one file, and a golden test that stays green, is the whole migration.

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now, so the board is current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check in (("exercise 1", _check_resolve),
                              ("exercise 2", _check_lifecycle),
                              ("exercise 3", _check_drift),
                              ("exercise 4", _check_normalise),
                              ("exercise 5", _check_golden)):
            _try(_name, _check)
    _progress_board()
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero. Inside a notebook kernel the
    # board above has already said so, in a line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.